In [32]:
!pip install -U langchain-classic langchain-google-genai langchain-community duckduckgo-search pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 79.0 MB/s eta 0:00:00


In [34]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.7 MB/s eta 0:00:00


In [35]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

In [97]:
from pydantic import BaseModel,Field
import re
from langchain_core.tools import tool
@tool
def get_usd_to_bdt_rate(query:str)->str:
  """Searches DuckDuckGo to find the current exchange rate from USD to BDT."""
  search_result=search_tool.run(query)

  return f"Search Results:{search_result}"

@tool
def get_usd_to_bdt_rate(query: str = "1 USD to BDT") -> str:
    """Searches DuckDuckGo to find the current exchange rate from USD to BDT."""
    search_results = search_tool.run(query)
    return f"Search Results: {search_results}"

@tool
def convert_usd_to_bdt(tool_input: str) -> str:
  """Calculates BDT value given USD amount and conversion rate separated by a comma.
  Example Action Input: 10, 123.50
  """
  # Extract floating-point numbers from whatever string format the LLM outputs
  numbers = re.findall(r"[-+]?\d*\.\d+|\d+", tool_input)
  if len(numbers) >= 2:
      usd_amount = float(numbers[0])
      conversion_rate = float(numbers[1])
      result = usd_amount * conversion_rate
      return f"{usd_amount} USD is equal to {result:.2f} BDT at rate {conversion_rate}"
  return "Error: Please provide both usd_amount and conversion_rate separated by a comma (e.g., '10, 123.50')."

tools = [get_usd_to_bdt_rate, convert_usd_to_bdt]


In [98]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [99]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_react_agent,AgentExecutor
from langchain_core.prompts import PromptTemplate

llm=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=GOOGLE_API_KEY
)

In [100]:
# ReAct Prompt Template
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [101]:
agent=create_react_agent(llm,tools,prompt)

agent_executor=AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [105]:
query = "Search DuckDuckGo to get the conversion rate for 1 USD to BDT, and then convert 10 USD to BDT using that rate."

response = agent_executor.invoke({"input": query})

print("\n--- Final Answer ---")
print(response["output"])



> Entering new AgentExecutor chain...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 51.166848084s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [ ]:
convert_usd_to_bdt.invoke(
    {
        "usd_amount": 10,
        "conversion_rate": 123.50
    }
)